# Real-Time Healthcare Patient Monitoring Data Platform

## Business Problem

Hospitals generate continuous patient-monitoring data from devices such as ECG monitors,
pulse oximeters, blood-pressure monitors, and temperature sensors.

The goal of this project is to build a scalable Databricks Lakehouse pipeline that:

- Ingests high-volume patient events
- Supports incremental processing
- Cleans and validates healthcare data
- Creates business-ready analytics datasets
- Identifies abnormal patient vital signs
- Provides operational monitoring
- Powers Power BI dashboards

## End-to-End Architecture

Patient Monitoring Devices
        ↓
JSON Patient Event Files
        ↓
Unity Catalog Volume
        ↓
Databricks Auto Loader
        ↓
Bronze Delta Table
        ↓
Silver Transformation
        ↓
Data Quality Validation
        ↓
Gold Business Tables
        ↓
Power BI Dashboard

Additional Enterprise Components:

- Auto Loader Checkpointing
- Schema Inference and Evolution
- Incremental Processing
- Delta OPTIMIZE
- ZORDER
- Pipeline Monitoring
- Error Logging
- Databricks Workflows
- Serverless Compute

## Medallion Architecture

### Bronze Layer

Table:
`workspace.healthcare_demo.bronze_patient_events`

Purpose:

The Bronze layer stores raw patient-event data ingested from JSON files using Databricks Auto Loader.

Key Features:

- Auto Loader using cloudFiles
- Schema inference
- Schema tracking
- Checkpointing
- Incremental ingestion
- Delta Lake storage


### Silver Layer

Table:
`workspace.healthcare_demo.silver_patient_events`

Purpose:

The Silver layer cleans and standardizes patient-event data.

Transformations include:

- Timestamp conversion
- Numeric type casting
- Healthcare data validation
- Patient data standardization
- Invalid-record detection


### Gold Layer

Tables:

- `gold_hospital_summary_clean`
- `gold_ward_summary`
- `gold_device_summary`
- `gold_patient_alerts`

Purpose:

The Gold layer contains business-ready datasets optimized for analytics and Power BI reporting.

## Data Volume

Source design:

- 200,000+ JSON files
- 5 patient events per file
- Approximately 1 million+ patient events generated

Incremental test:

- New files were added after the initial ingestion
- Auto Loader checkpointing detected only newly arrived files
- Bronze and Silver tables were incrementally updated

Current pipeline validation demonstrated more than 1.2 million Bronze/Silver records during incremental testing.

## Performance Optimization

### Delta OPTIMIZE

OPTIMIZE was applied to Bronze, Silver, and Gold Delta tables to compact small files
and improve query performance.

### ZORDER

The patient-alert Gold table was Z-Ordered using:

hospital_id
patient_id

This improves data skipping for common hospital and patient filtering queries.

### Serverless Compute

Databricks Serverless compute was used for pipeline execution.

Benefits:

- Automatic infrastructure provisioning
- Automatic scaling
- Reduced cluster-management overhead
- Faster development workflow

## Monitoring and Error Handling

A Delta monitoring table was created:

`workspace.healthcare_demo.pipeline_execution_log`

The monitoring framework captures:

- Pipeline name
- Processing layer
- Start time
- End time
- Records processed
- Execution status
- Execution duration
- Error message

Both SUCCESS and FAILED executions were tested.

Failure testing successfully captured a TABLE_OR_VIEW_NOT_FOUND exception
inside the monitoring table.

## Databricks Workflow

Workflow Name:

`Healthcare_Data_Pipeline`

Execution order:

Bronze_Ingestion
        ↓
Silver_Transformation
        ↓
Gold_Analytics
        ↓
Pipeline_Monitoring

Databricks Workflows manages task dependencies and executes the pipeline
from ingestion through analytics and monitoring.

## Power BI Dashboard

Power BI connects to Databricks through the Databricks SQL Warehouse.

Dashboard datasets:

- Hospital Summary
- Ward Summary
- Device Summary
- Patient Alerts

Dashboard KPIs:

- Total Patient Events
- Total Alerts
- Average Heart Rate
- Average SpO2

Dashboard Visualizations:

- Hospital-wise Event Distribution
- Ward-wise Distribution
- Device-wise Event Analysis
- Critical Patient Alerts
- Interactive Hospital, Ward, and Device Filters

Medical / IoT Devices
        ↓
200K+ JSON Files
        ↓
Unity Catalog Volume
        ↓
Auto Loader + Checkpoint
        ↓
Bronze Delta
        ↓
Silver Cleaning + DQ
        ↓
Gold Analytics
   ├── Hospital Summary
   ├── Ward Summary
   ├── Device Summary
   └── Patient Alerts
        ↓
Databricks SQL Warehouse
        ↓
Power BI

        PLUS

Databricks Workflows
Pipeline Monitoring
Error Logging
OPTIMIZE
ZORDER
Serverless Compute